# Runge preprocessing

In [1]:
# Settings and preprocessing of Runge function
# Ensures that same settings and preprocessing are used in both part B and C

# imports
import autograd.numpy as np 
from sklearn.model_selection import train_test_split

# custom imports
from scaling import standard_scaler


## --- Settings --- 
# Constants
DATAPOINTS = 1000
STANDARD_DEVIATION = 0.1

TEST_SPLIT = 0.2
TRAIN_SPLIT = 1 - TEST_SPLIT

TEST_TRAIN_RANDOM_STATE = 42 # ensure reproducibility train_test_split
NP_RANDOM_SEED = 250 # ensure reproducibility numpy

ETA_VALUES = [0.1, 0.01, 0.001, 0.0001]
LAMBDA_VALUES = np.logspace(-2, -4, 10)

VERBOSE = False

RUNGE_HIDDEN_LAYERS = (50, 100)
RUNGE_MAX_ITERATIONS = 10


# Defining Runge function 
def runge_function(x, n_datapoints=DATAPOINTS, standard_deviation=STANDARD_DEVIATION):
    y = 1 / (1 + 25 * x**2) + np.random.normal(0, standard_deviation, n_datapoints)
    return y



## --- Preprocessing --- 
# Generate data for Runge function
x = np.linspace(-1, 1, DATAPOINTS)
np.random.seed(NP_RANDOM_SEED)
y_noise = runge_function(x)
np.random.seed(NP_RANDOM_SEED)
y = runge_function(x, n_datapoints=DATAPOINTS, standard_deviation=0) # override standard deviation to get true function



# check to test back propagation and gradient computation in part c with more features
test_two_features = False
two_features_two_predictors = False


if test_two_features:
    np.random.seed(27) # use different seed for different numbers
    noise = np.random.normal(0, 0.13, x.shape)
    x_noisy = x + noise
    x_noise = np.vstack((x, x_noisy)).T
    x = x_noise

if two_features_two_predictors:
    y_noise2 = runge_function(x, standard_deviation=0.07) # and different standard deviation
    y_noise = np.vstack((y_noise, y_noise2)).T
    np.random.seed(27) # use different seed for different numbers
    x2 = np.linspace(-1, 1, DATAPOINTS)
    noise = np.random.normal(0, 0.13, x.shape)
    x_noisy = x + noise

    x_noise = np.vstack((x, x2)).T


    x = x_noise


# preprosessing data
x_train, x_test, y_train, y_test = train_test_split(x, y_noise, test_size=TEST_SPLIT, random_state=TEST_TRAIN_RANDOM_STATE)

# scaling of x_train and x_test
x_train_scaled, x_test_scaled, x_train_mean, x_train_std = standard_scaler(x_train, x_test) # --> verified too give same results as sklearn StandardScaler for x_train



print('Before reshaping')
print('x', x.shape)
print('y_noise', y_noise.shape)

print('x_train', x_train.shape)
print('y_train', y_train.shape)


print('x_train_scaled', x_train_scaled.shape)
print('x_test_scaled', x_test_scaled.shape)
print('y_train', y_train.shape)


# reshaping
if test_two_features:
    y_train = np.array(y_train).reshape(-1, 1)
    y_test = np.array(y_test).reshape(-1, 1)
    y_noise = np.array(y_noise).reshape(-1, 1)
elif two_features_two_predictors:
    print('All good')
else:
    # Reshape for use in neural network code
    x_train_scaled = np.array(x_train_scaled).reshape(-1,1)     
    x_test_scaled = np.array(x_test_scaled).reshape(-1,1)
    y_train = np.array(y_train).reshape(-1, 1)
    y_test = np.array(y_test).reshape(-1, 1)

print('After reshaping')
print('x', x.shape)
print('y_noise', y_noise.shape)

print('x_train', x_train.shape)
print('y_train', y_train.shape)


print('x_train_scaled', x_train_scaled.shape)
print('x_test_scaled', x_test_scaled.shape)
print('y_train', y_train.shape)



Before reshaping
x (1000,)
y_noise (1000,)
x_train (800,)
y_train (800,)
x_train_scaled (800,)
x_test_scaled (200,)
y_train (800,)
After reshaping
x (1000,)
y_noise (1000,)
x_train (800,)
y_train (800, 1)
x_train_scaled (800, 1)
x_test_scaled (200, 1)
y_train (800, 1)


# Testing code

In [2]:
from activation_functions import sigmoid, linear
from cost_functions import mse

# Settings for this task
network_input_size = x_train_scaled.shape[1]   # ----> should be the same as number of features
output_dim = 1 if y_test.ndim == 1 else y_test.shape[1] 
layer_output_sizes = [*RUNGE_HIDDEN_LAYERS, output_dim]

# create activation functions list - sigmoid and end with linear for regression
num_layers = len(layer_output_sizes)


activation_functions_object = [sigmoid.sigmoid_func] * (num_layers - 1) + [linear.linear] 
activation_functions_derivatives_object = [sigmoid.sigmoid_derivative] * (num_layers - 1) + [linear.linear_derivative]

# sanity check, print shapes - REMOVE LATER
print(x_train_scaled.shape, y_test.shape)
print(layer_output_sizes)
print(activation_functions_object)

print(network_input_size)



(800, 1) (200, 1)
[50, 100, 1]
[<function sigmoid.sigmoid_func at 0x0000020205059800>, <function sigmoid.sigmoid_func at 0x0000020205059800>, <function linear.linear at 0x0000020205059C60>]
1


In [3]:
from neural_network import NN

In [5]:

test = NN(dims = [network_input_size, 50, 100, 1], activation_funcs = activation_functions_object, activation_ders = activation_functions_derivatives_object)

test._feedforward(X = x_train_scaled)
object_gradients = test._backpropagation(inputs = x_train_scaled, targets = y_train)

print(object_gradients)
for i in object_gradients:
    print(i[0].shape)


[(array([[ 0.51634189,  0.19584643,  3.36108224, -0.04201322, -0.50247104,
         0.95245944,  0.79706053,  0.55376733,  0.39851006,  0.94807285,
        -1.35178179,  0.89107278, -0.1216019 ,  1.83200089, -1.15615991,
         0.77382023, -1.35237143,  1.03065886,  0.05857038, -0.15119287,
         0.16913304, -2.41826847,  0.82436127, -0.475466  ,  0.15940301,
         0.77900363,  2.110576  , -1.68660789, -0.02873717, -0.21126151,
         0.5616236 , -1.83222948, -0.67751758, -0.51757356, -0.07069799,
         0.01279693,  0.87668498,  0.06777612, -0.07139338,  1.2994474 ,
         0.39472827, -0.14021746,  0.25818577,  0.54796894, -0.91242507,
         0.07793024,  3.53514509, -1.2927365 ,  0.45883383, -0.08829182]]), array([-0.32363216, -0.62670117, -4.70685448,  0.34858969,  0.53764022,
       -0.98554339, -2.54461178, -0.33869287, -0.40178497, -2.62327769,
        1.92860317, -1.46866038,  1.18589674, -3.73373127,  1.02796214,
       -1.09738066,  1.82308776, -2.17699231, -1.